In [1]:
%%capture
!pip install peft
!pip install evaluate
!pip install datasets
!pip install "transformers==4.57.2"
!pip install sentencepiece
!pip install emoji

In [2]:
import os
import shutil
import re
import emoji
import logging

import evaluate
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from transformers import (
    DebertaV2ForSequenceClassification,
    DebertaV2Tokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    AutoTokenizer, 
    AutoModelForSequenceClassification
)

In [3]:
def clean_text(text):
    if pd.isna(text):
        return text

    # 1. lowercase
    text = text.lower()

    # 2. remove @USER mentions
    text = re.sub(r'@user', '', text, flags=re.IGNORECASE)
    text = re.sub(r'@url', '', text, flags=re.IGNORECASE)

    # 3. remove URLs (actual links or placeholder "URL")
    # text = re.sub(r'http\S+|https\S+|url', '', text, flags=re.IGNORECASE)

    # 4. remove underscores, repeated underscores
    text = re.sub(r'_+', ' ', text)

    # 5. remove slashes
    text = text.replace('\\', ' ').replace('/', ' ')

    # 6. remove emojis
    text = emoji.replace_emoji(text, replace="")

    # 7. remove quotation marks (normal + smart)
    text = re.sub(r"[\"“”]", "", text)

    # 8. normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [4]:
# Setup
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [5]:
%%time
# Load Model & Tokenizer
MODEL_NAME = "microsoft/mdeberta-v3-base"

try:
    tokenizer = DebertaV2Tokenizer.from_pretrained(MODEL_NAME)
    base_model = DebertaV2ForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
except Exception as e:
    print(f"Error loading DebertaV2: {e}. Trying AutoClasses...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

base_model.to(device)

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/579 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/mdeberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


CPU times: user 4.38 s, sys: 2.98 s, total: 7.35 s
Wall time: 5.76 s


DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(251000, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): Layer

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

In [6]:
# Load Data
DATA_DIR = "subtask1"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
DEV_DIR = os.path.join(DATA_DIR, "dev")
TEST_DIR = os.path.join(DATA_DIR, "test")

def load_split(split_dir):
    dfs = []
    if not os.path.exists(split_dir):
        print(f"Directory not found: {split_dir}")
        return pd.DataFrame()
    for file in os.listdir(split_dir):
        if file.endswith(".csv"):
            lang = file.replace(".csv", "")
            df = pd.read_csv(os.path.join(split_dir, file))
            df["lang"] = lang
            dfs.append(df)
    if not dfs:
        return pd.DataFrame()
    return pd.concat(dfs, ignore_index=True)

print("Loading Train Data...")
raw_train_df = load_split(TRAIN_DIR)
print(f"Loaded {len(raw_train_df)} training examples")

print("Loading Dev Data (Used as internal Test)...")
raw_dev_df = load_split(DEV_DIR)
print(f"Loaded {len(raw_dev_df)} dev examples")

print("Loading Test Data (For Submission)...")
raw_test_df = load_split(TEST_DIR)
print(f"Loaded {len(raw_test_df)} test examples")

Loading Train Data...
Loaded 73681 training examples
Loading Dev Data (Used as internal Test)...
Loaded 3687 dev examples
Loading Test Data (For Submission)...
Loaded 33288 test examples


In [7]:
# Preprocess Text
print("Preprocessing text (cleaning)...")
raw_train_df["text"] = raw_train_df["text"].astype(str).apply(clean_text)
raw_dev_df["text"] = raw_dev_df["text"].astype(str).apply(clean_text)
raw_test_df["text"] = raw_test_df["text"].astype(str).apply(clean_text)

Preprocessing text (cleaning)...


In [8]:
# Data Splitting
# Rename 'polarization' to 'labels'
if "polarization" in raw_train_df.columns:
    raw_train_df = raw_train_df.rename(columns={"polarization": "labels"})
if "polarization" in raw_dev_df.columns:
    raw_dev_df = raw_dev_df.rename(columns={"polarization": "labels"})

# Split Train into 95% Train / 5% Val
train_df, val_df = train_test_split(
    raw_train_df,
    test_size=0.05,
    stratify=raw_train_df["labels"],
    random_state=SEED,
    shuffle=True
)

# Use Dev as Internal Test
test_df = raw_dev_df.copy()

print("Shape after split:")
print(f"Train:      {train_df.shape}")
print(f"Validation: {val_df.shape}")
print(f"Test (Dev): {test_df.shape}")

Shape after split:
Train:      (69996, 4)
Validation: (3685, 4)
Test (Dev): (3687, 4)


In [9]:
# Create Dataset Objects
train_dataset = Dataset.from_pandas(train_df[["text", "labels"]], preserve_index=False)
val_dataset = Dataset.from_pandas(val_df[["text", "labels"]], preserve_index=False)
test_dataset = Dataset.from_pandas(test_df[["text", "labels"]], preserve_index=False)

dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

In [10]:
%%time
# Tokenize
def tokenize_function(examples):
    return tokenizer(
        examples["text"], 
        padding="max_length", 
        truncation=True, 
        max_length=256
    )

print("Tokenizing datasets...")
encoded_dataset = dataset.map(tokenize_function, batched=True)
encoded_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

Tokenizing datasets...


Map:   0%|          | 0/69996 [00:00<?, ? examples/s]

Map:   0%|          | 0/3685 [00:00<?, ? examples/s]

Map:   0%|          | 0/3687 [00:00<?, ? examples/s]

CPU times: user 13.7 s, sys: 257 ms, total: 14 s
Wall time: 14 s


In [11]:
# Training Arguments
OUTPUT_DIR = "./output_results"
BATCH_SIZE = 32
EPOCHS = 30
LR = 2e-5
GRAD_ACCUM = 2

# Calculate steps
steps_per_epoch = len(encoded_dataset["train"]) // (BATCH_SIZE * GRAD_ACCUM)
eval_steps = steps_per_epoch
steps_per_epoch

1093

In [12]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_steps=1000,
    gradient_accumulation_steps=GRAD_ACCUM,
    logging_steps=eval_steps,
    eval_steps=eval_steps,
    save_steps=eval_steps * 10,  # Save less frequently to save space
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    eval_strategy="steps",
    logging_dir=f"{OUTPUT_DIR}/logs",
    report_to="none",
    fp16=torch.cuda.is_available(),
)

metric_f1 = evaluate.load("f1")
metric_acc = evaluate.load("accuracy")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="macro")["f1"]
    acc = metric_acc.compute(predictions=predictions, references=labels)["accuracy"]
    return {"f1": f1, "accuracy": acc}

trainer = Trainer(
    model=base_model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

In [13]:
%%time
# Start Training
trainer.train()

Step,Training Loss,Validation Loss,F1,Accuracy
1093,0.543800,0.435253,0.796798,0.798100
2186,0.403300,0.387326,0.822508,0.823881
3279,0.322200,0.396113,0.827211,0.828223
4372,0.255900,0.434464,0.823736,0.824152
5465,0.197600,0.469804,0.820674,0.821710
6558,0.150600,0.649256,0.814383,0.814383


CPU times: user 32min 44s, sys: 1min 18s, total: 34min 2s
Wall time: 34min 7s


TrainOutput(global_step=6558, training_loss=0.3122254488271124, metrics={'train_runtime': 2047.5479, 'train_samples_per_second': 1025.558, 'train_steps_per_second': 16.029, 'total_flos': 5.520326806977331e+16, 'train_loss': 0.3122254488271124, 'epoch': 5.994515539305302})

In [14]:
# Evaluation on Internal Test Set (Dev Folder)
print("Evaluating on Internal Test Set (Dev folder data)...")
preds_output = trainer.predict(encoded_dataset["test"])

pred_labels = np.argmax(preds_output.predictions, axis=1)
true_labels = preds_output.label_ids

print("\nClassification Report:")
report = classification_report(true_labels, pred_labels, target_names=["Not Polar (0)", "Polar (1)"], digits=4)
print(f"\n{report}")

macro_f1 = f1_score(true_labels, pred_labels, average='macro')
print(f"Macro F1: {macro_f1:.4f}")

# Per-Language Analysis
test_df["preds"] = pred_labels
print("\n=== Macro F1 per Language ===")
results = []
for lang in sorted(test_df["lang"].unique()):
    lang_df = test_df[test_df["lang"] == lang]
    f1 = f1_score(lang_df["labels"], lang_df["preds"], average="macro")
    acc = accuracy_score(lang_df["labels"], lang_df["preds"])
    print(f"{lang}: F1={f1:.4f}, Acc={acc:.4f}, Support={len(lang_df)}")
    results.append({"lang": lang, "f1_macro": f1, "accuracy": acc, "count": len(lang_df)})

results_df = pd.DataFrame(results)
print(f"\nAverage Macro F1 across languages: {results_df['f1_macro'].mean():.4f}")

Evaluating on Internal Test Set (Dev folder data)...



Classification Report:

               precision    recall  f1-score   support

Not Polar (0)     0.7735    0.8538    0.8117      1744
    Polar (1)     0.8553    0.7756    0.8135      1943

     accuracy                         0.8126      3687
    macro avg     0.8144    0.8147    0.8126      3687
 weighted avg     0.8166    0.8126    0.8126      3687

Macro F1: 0.8126

=== Macro F1 per Language ===
amh: F1=0.6923, Acc=0.7470, Support=166
arb: F1=0.7539, Acc=0.7633, Support=169
ben: F1=0.8373, Acc=0.8434, Support=166
deu: F1=0.6552, Acc=0.6792, Support=159
eng: F1=0.7173, Acc=0.7625, Support=160
fas: F1=0.8859, Acc=0.9085, Support=164
hau: F1=0.6731, Acc=0.9011, Support=182
hin: F1=0.8193, Acc=0.8905, Support=137
ita: F1=0.6525, Acc=0.6747, Support=166
khm: F1=0.6542, Acc=0.9217, Support=332
mya: F1=0.8952, Acc=0.8958, Support=144
nep: F1=0.8400, Acc=0.8400, Support=100
ori: F1=0.7037, Acc=0.7881, Support=118
pan: F1=0.7895, Acc=0.7900, Support=100
pol: F1=0.7810, Acc=0.7899, Suppor

In [15]:
# Generate Submission (Test Folder)
SUBMISSION_DIR = "./subtask_1"
if os.path.exists(SUBMISSION_DIR):
    shutil.rmtree(SUBMISSION_DIR)
os.makedirs(SUBMISSION_DIR)

print("Generating predictions for submission...")

# tokenize test set for submission
submission_dataset = Dataset.from_pandas(raw_test_df[["text"]], preserve_index=False)
submission_tokenized = submission_dataset.map(tokenize_function, batched=True)
submission_tokenized.set_format(type="torch", columns=["input_ids", "attention_mask"])

# Predict
submission_preds_output = trainer.predict(submission_tokenized)
submission_labels = np.argmax(submission_preds_output.predictions, axis=1)

# Add predictions back to dataframe
raw_test_df["polarization"] = submission_labels

# Save individual files
languages = sorted(raw_test_df["lang"].unique())
print(f"Processing {len(languages)} languages for submission...")

for lang in languages:
    lang_df = raw_test_df[raw_test_df["lang"] == lang]
    output_df = lang_df[["id", "polarization"]]
    
    output_path = os.path.join(SUBMISSION_DIR, f"pred_{lang}.csv")
    output_df.to_csv(output_path, index=False)
    
print("Zipping prediction files...")
shutil.make_archive("subtask_1", "zip", SUBMISSION_DIR)
print(f"Created subtask_1.zip in {os.getcwd()}")

Generating predictions for submission...


Map:   0%|          | 0/33288 [00:00<?, ? examples/s]

Processing 22 languages for submission...
Zipping prediction files...
Created subtask_1.zip in /home/jovyan/work
